In [1]:
import os
from dotenv import load_dotenv
import random
from datasets import load_dataset
from collections import Counter
load_dotenv() 

/home/prateek/Desktop/llm_with_ads/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# Arena Human Preference 140k (~136k rows, ~1.6GB)
# https://huggingface.co/datasets/lmarena-ai/arena-human-preference-140k
ds = load_dataset(
    "lmarena-ai/arena-human-preference-140k",
    token=os.getenv("HF_TOKEN"),
)


In [3]:
train = ds["train"]
print(f"Rows: {len(train):,}")
train.features

Rows: 135,634


{'id': Value('string'),
 'model_a': Value('string'),
 'model_b': Value('string'),
 'winner': Value('string'),
 'evaluation_session_id': Value('string'),
 'evaluation_order': Value('int32'),
 'conversation_a': List({'role': Value('string'), 'content': List({'type': Value('string'), 'text': Value('string'), 'image': Value('string'), 'mimeType': Value('string')})}),
 'conversation_b': List({'role': Value('string'), 'content': List({'type': Value('string'), 'text': Value('string'), 'image': Value('string'), 'mimeType': Value('string')}), 'num_tokens': Value('int32')}),
 'full_conversation': List({'user': {'role': Value('string'), 'content': List({'type': Value('string'), 'text': Value('string'), 'image': Value('string'), 'mimeType': Value('string')})}, 'model_side_a': {'role': Value('string'), 'content': List({'type': Value('string'), 'text': Value('string'), 'image': Value('string'), 'mimeType': Value('string')})}, 'model_side_b': {'role': Value('string'), 'content': List({'type': Value('

In [4]:
# Peek at one row (omit long conversation bodies for readability)
row = train[0]
{k: row[k] for k in ("id", "model_a", "model_b", "winner", "language", "is_code", "timestamp")}

{'id': 'c4b9710c-8d64-4bee-a0b0-94637ae4cc65',
 'model_a': 'gemini-2.5-pro',
 'model_b': 'claude-3-7-sonnet-20250219-thinking-32k',
 'winner': 'model_a',
 'language': 'en',
 'is_code': False,
 'timestamp': Timestamp('2025-05-29 14:48:41.602925')}

In [5]:
# Keep only non-code conversations
train_no_code = train.filter(lambda x: not x["is_code"])
print(f"Rows after filtering out is_code=True: {len(train_no_code):,}")


Rows after filtering out is_code=True: 96,271


In [6]:
# Keep only English conversations
train_only_en = train_no_code.filter(lambda x: x["language"] == "en")
print(f"Rows after filtering out non-English: {len(train_only_en):,}")


Rows after filtering out non-English: 46,809


In [7]:
set(train_only_en["winner"])

{'both_bad', 'model_a', 'model_b', 'tie'}

In [8]:
train_only_tie = train_only_en.filter(lambda x: x["winner"] == 'tie')
len(train_only_tie)

train_only_both_bad = train_only_en.filter(lambda x: x["winner"] == 'both_bad')
len(train_only_both_bad)



5152

In [9]:
# Drop both_bad; keep ties (pick a side at random below)
n_before = len(train_only_en)
train_no_both_bad = train_only_en.filter(lambda x: x["winner"] != "both_bad")
print(f"Dropped both_bad: {n_before - len(train_no_both_bad):,} | remaining: {len(train_no_both_bad):,}")


def add_winner_cols(row):
    winner = row["winner"]
    if winner == "model_a":
        side = "a"
    elif winner == "model_b":
        side = "b"
    else:  # tie — deterministic random choice from id
        side = random.Random(row["id"]).choice(["a", "b"])

    if side == "a":
        model = row["model_a"]
        conv = row["conversation_a"]
    else:
        model = row["model_b"]
        conv = row["conversation_b"]

    # Normalize schema (conversation_b has optional num_tokens)
    conversation = [{"role": m["role"], "content": m["content"]} for m in conv]
    return {"model": model, "conversation": conversation}


train_filtered = train_no_both_bad.map(add_winner_cols)
train_filtered.select_columns(["id", "winner", "model", "conversation"])[0]


Dropped both_bad: 5,152 | remaining: 41,657


{'id': 'c4b9710c-8d64-4bee-a0b0-94637ae4cc65',
 'winner': 'model_a',
 'model': 'gemini-2.5-pro',
 'conversation': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'Compare Tormenta20 with DnD5e',
     'image': None,
     'mimeType': None}]},
  {'role': 'assistant',
   'content': [{'type': 'text',
     'text': 'Excellent question! Comparing **Tormenta20 (T20)** and **Dungeons & Dragons 5th Edition (D&D 5e)** is a fantastic way to understand different approaches to the d20 system. While they share a common ancestor (D&D 3rd Edition), they have evolved in vastly different directions.\n\nHere is a detailed comparison, broken down by key aspects of the games.\n\n---\n\n### **High-Level Philosophy**\n\n*   **D&D 5e:** Aims for **accessibility, streamlined play, and heroic fantasy**. It simplifies many of the complex rules of older editions to be welcoming to new players. Its design philosophy is "rulings, not rules," encouraging Dungeon Masters to make calls on the fly. It is b

In [10]:
from collections import defaultdict

# Unique content types per role in the winning conversation
types_by_role = defaultdict(set)

for conv in train_filtered["conversation"]:
    for msg in conv:
        role = msg["role"]
        for part in msg["content"]:
            types_by_role[role].add(part["type"])

{role: sorted(types) for role, types in types_by_role.items()}


{'user': ['text'], 'assistant': ['text']}

In [11]:
# Max number of conversation message blocks in a single row
lengths = [len(conv) for conv in train_filtered["conversation"]]
print(f"max blocks: {max(lengths)}")
print(f"min blocks: {min(lengths)}")
print(f"mean blocks: {sum(lengths) / len(lengths):.2f}")


max blocks: 128
min blocks: 2
mean blocks: 2.42


In [12]:
import matplotlib.pyplot as plt
from collections import Counter

counts = Counter(lengths)
counts


Counter({2: 36579,
         4: 3447,
         6: 907,
         8: 326,
         10: 170,
         12: 89,
         16: 33,
         14: 32,
         18: 19,
         20: 14,
         22: 11,
         30: 5,
         26: 5,
         24: 5,
         28: 4,
         74: 2,
         38: 2,
         36: 2,
         32: 2,
         128: 1,
         40: 1,
         56: 1})

In [13]:
# Example row with exactly 4 conversation blocks
idx = next(i for i, n in enumerate(lengths) if n == 4)
row = train_filtered[idx]

print({k: row[k] for k in ("id", "winner", "model", "language")})
print(f"blocks: {len(row['conversation'])}")
for i, msg in enumerate(row["conversation"]):
    text = msg["content"][0]["text"] if msg["content"] else ""
    preview = text[:300] + ("..." if len(text) > 300 else "")
    print(f"\n[{i}] role={msg['role']}\n{preview}")


{'id': '3636b7c2-4efe-4c29-8ce9-713cdcdefdc5', 'winner': 'tie', 'model': 'qwq-32b', 'language': 'en'}
blocks: 4

[0] role=user
So it didn't reach being significantly usable. What are promising ways it could be done with much less noise? Or maybe the way is creating quantum programs that denoise and scaling noisy quantum computers? BTW, why none of existing quantum computers can modify and extend themselves? It is quantum af...

[1] role=assistant
Your questions touch on some of the most critical challenges and opportunities in quantum computing. Let’s break down the key points systematically:

---

### **1. Current Quantum Computers Are Not Yet "Significantly Usable"**
Quantum computers today operate in the **Noisy Intermediate-Scale Quantum...

[2] role=user
Which computers with corresponding software, other than LLM agent that writes HDL and requests foundry to produce that, can improve their hardware?

[3] role=assistant
The concept of a computer system that can **autonomously impro

In [14]:
train_small_conv = train_filtered.filter(lambda x: len(x["conversation"]) == 2)
len(train_small_conv)

36579

In [15]:
def _text_from_msg(msg):
    return "".join(part["text"] or "" for part in msg["content"] if part["type"] == "text")


def add_query_response(row):
    query = ""
    llm_response = ""
    for msg in row["conversation"]:
        if msg["role"] == "user":
            query = _text_from_msg(msg)
        elif msg["role"] == "assistant":
            llm_response = _text_from_msg(msg)
    return {"query": query, "llm_response": llm_response}


train_small_conv = train_small_conv.map(add_query_response)
train_small_conv.select_columns(["id", "model", "query", "llm_response"])[0]


{'id': 'c4b9710c-8d64-4bee-a0b0-94637ae4cc65',
 'model': 'gemini-2.5-pro',
 'query': 'Compare Tormenta20 with DnD5e',
 'llm_response': 'Excellent question! Comparing **Tormenta20 (T20)** and **Dungeons & Dragons 5th Edition (D&D 5e)** is a fantastic way to understand different approaches to the d20 system. While they share a common ancestor (D&D 3rd Edition), they have evolved in vastly different directions.\n\nHere is a detailed comparison, broken down by key aspects of the games.\n\n---\n\n### **High-Level Philosophy**\n\n*   **D&D 5e:** Aims for **accessibility, streamlined play, and heroic fantasy**. It simplifies many of the complex rules of older editions to be welcoming to new players. Its design philosophy is "rulings, not rules," encouraging Dungeon Masters to make calls on the fly. It is built to be setting-agnostic, though the Forgotten Realms is its default.\n*   **Tormenta20:** Aims for **deep character customization, high-power fantasy, and an "anime/JRPG" feel**. It offe

In [17]:
from collections import Counter

# Group by model family prefix (text before first "-")
prefixes = [m.split("-", 1)[0] for m in train_small_conv["model"]]
rows_per_model = Counter(prefixes)
sorted(rows_per_model.items(), key=lambda x: (-x[1], x[0]))


[('gemini', 6480),
 ('claude', 6377),
 ('qwen3', 2848),
 ('grok', 2471),
 ('llama', 2261),
 ('o3', 2124),
 ('deepseek', 2031),
 ('mistral', 1878),
 ('gpt', 1862),
 ('gemma', 1348),
 ('chatgpt', 1289),
 ('o4', 914),
 ('minimax', 884),
 ('command', 809),
 ('qwen', 678),
 ('amazon.nova', 622),
 ('qwq', 621),
 ('kimi', 361),
 ('amazon', 335),
 ('hunyuan', 207),
 ('magistral', 179)]